# Assignment 04

In [1]:
# Load modules
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Import

In [5]:
# Load data from CSV file
df = pd.DataFrame()
df = pd.read_csv('data.csv', dtype={'Radius (cm)': float, 'Weight (grams)': float})
print(df)

# Replace zero values with median value within class 
df = df.groupby(['Fruit (class)']) # sort by class
proc_df = pd.DataFrame() # create a new dataframe to collect results
for key, group in df: # go over groups
    group = group.replace(0, group.median(axis=0)) # replace missing values with median
    proc_df = pd.concat([proc_df, group]) # concatenate groups into new dataframe
df = proc_df.sort_index() # overwrite original dataframe with results

# Normalize
radius = df['Radius (cm)']
df['Radius (cm)'] = (radius-radius.min())/(radius.max()-radius.min())
weight = df['Weight (grams)']
df['Weight (grams)'] = (weight-weight.min())/(weight.max()-weight.min())
print(df)

   Radius (cm)  Weight (grams) Fruit (class)
0         65.0           325.0         Lemon
1         68.0           350.0         Apple
2         87.0           312.0         Lemon
3         77.0           324.0         Apple
4         73.0           300.0         Lemon
5         90.0           370.0         Apple
6         61.0           365.0         Apple
7         62.0           400.0         Apple
8         92.0           340.0         Lemon
   Radius (cm)  Weight (grams) Fruit (class)
0     0.129032            0.25         Lemon
1     0.225806            0.50         Apple
2     0.838710            0.12         Lemon
3     0.516129            0.24         Apple
4     0.387097            0.00         Lemon
5     0.935484            0.70         Apple
6     0.000000            0.65         Apple
7     0.032258            1.00         Apple
8     1.000000            0.40         Lemon


In [ ]:
# Radius values
radius = list(df['Radius (cm)'])
print(radius)

In [ ]:
# Weight values
weight = list(df['Weight (grams)'])
print(weight)

In [ ]:
# Classes
classes = list(df['Fruit (class)'])
print(classes)

In [ ]:
# Combine radius, weight, and class to tuples (we have to keep the class for later plotting)
tuples = list(zip(radius, weight, classes))
print(tuples)

## Clustering

In [ ]:
# Squared Euclidean distance (use this as delta function)
def distance(p1,p2):
    dist = (p2[0]-p1[0])**2+(p2[1]-p1[1])**2
    return dist

# Takes points as list of tuples and a threshold.
# Example call: do_cluster([(2,1,'Apple'),(6,3,'Lemon'),(1,1.5,'Apple'),(2,2,'Pear')], 7)
def do_cluster(tuples, threshold):
    clusters = [[],[]] 
    #Initialising first object of tuple as centroid and rest as 0 
    clusterCentroid=[[tuples[0][0],tuples[0][1]],[0,0]]
    clusterCentroidNew=[]
    #Putting first object from tuple in first cluster
    clusters[0].append(tuples[0])
    clusterNew=[]
    #0th object is already put in cluster
    for i in range(1,len(tuples)):
        if(distance(tuples[i],clusterCentroid[0])<=threshold):
            #appending particular object to first cluster
            clusters[0].append(tuples[i])
            lenCluster=len(clusters[0])
            #updating centroid for first cluster
            clusterCentroid[0]=[((tuples[i][0]/lenCluster) + (((lenCluster-1)/lenCluster)*clusterCentroid[0][0])),
                             ((tuples[i][1]/lenCluster) + (((lenCluster-1)/lenCluster)*clusterCentroid[0][1]))]
        else:
            if(len(clusters[1])==0):
                #putting object to second cluster if seconf cluster  is empty and initialising centroid for second cluster
                clusterCentroid[1]=[tuples[i][0],tuples[i][1]]
                clusters[1].append(tuples[i])
            else:
                if(distance(tuples[i],clusterCentroid[1])<=threshold):
                    clusters[1].append(tuples[i])
                    lenCluster=len(clusters[1])
                    #updating cluster centroid for second cluster
                    clusterCentroid[1]=[((tuples[i][0]/lenCluster)) + ((lenCluster-1)/lenCluster)*clusterCentroid[1][0],
                             ((tuples[i][1]/lenCluster) + (((lenCluster-1)/lenCluster)*clusterCentroid[1][1]))]
                else:
                    clusterNew.append(tuples[i])
                    #Updating cluster centroid for objects which doesnt fit in any cluster
                    clusterCentroidNew=[tuples[i][0],tuples[i][1]]
                    clusterCentroid.append(clusterCentroidNew)
    clusters.append(clusterNew)
    return clusters # e.g., [[(2, 1, 'Apple'), (1, 1.5, 'Apple'), (2, 2, 'Pear')], [(6, 3, 'Lemon')]]

# Call to cluster

clusters = do_cluster(tuples, 0.33) # distance threshold, aka Delta, is set to 0.33
print(clusters)

# Hint: Final clusters should look like the following:
# [[(0.06451612903225798, 0.15037593984962405, 'Lemon'), (0.16129032258064516, 0.13533834586466165, 'Lemon'), ...

## Plotting

In [ ]:
colors = ['red', 'green', 'blue', 'yellow', 'purple', 'orange'] # provide some colors for the clusters
marker = {'Lemon': '*', 'Apple': 'o', 'Pear': 'x'} # different marker for each class
i = 0
for c in clusters:
    tpls = list(zip(*c))
    x = tpls[0]
    y = tpls[1]
    cls = tpls[2]
    c = colors[i%len(colors)]
    m = [marker[cl] for cl in cls]
    for _x, _y, _m in zip(x, y, m):
        plt.scatter(_x, _y, c=c, marker=_m)
    i += 1
plt.xlabel("Radius")
plt.ylabel("Weight")
plt.show()